# leaf-tensor-condition — worked example 1: ARENA leaf condition: recipe is None and requires_grad is True

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `leaf-tensor-condition`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In the ARENA custom autograd framework, a MiniTensor is a leaf when its `recipe` attribute is `None` — meaning it was created by the user, not produced by a tracked operation. The full leaf condition also checks `requires_grad=True`, because a frozen tensor (no recipe, no grad) is also leaf-shaped but not trainable. Leaf tensors accumulate `.grad` rather than passing it through.

## Worked solution

Step 1: Define `is_leaf(tensor)` as `tensor.recipe is None and tensor.requires_grad`.

Step 2: Build four MiniTensors that cover the four combinations: (recipe=None, rg=True), (recipe=None, rg=False), (recipe=not None, rg=True), (recipe=not None, rg=False).

Step 3: Apply `is_leaf` to each. Only the first (recipe=None AND requires_grad=True) should return True.

Step 4: Print the results to confirm the logic.

In [ ]:
import numpy as np
from dataclasses import dataclass
from typing import Any, Optional

@dataclass
class Recipe:
    func: Any
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe: Optional[Recipe] = None

def is_leaf(tensor: MiniTensor) -> bool:
    """True iff the tensor is a user-created trainable leaf."""
    return tensor.recipe is None and tensor.requires_grad

arr = np.array([1.0, 2.0])

# Case 1: leaf-trainable (recipe=None, rg=True)
t1 = MiniTensor(arr.copy(), requires_grad=True)

# Case 2: leaf-frozen (recipe=None, rg=False)
t2 = MiniTensor(arr.copy(), requires_grad=False)

# Fake recipe for cases 3 and 4
fake_recipe = Recipe(np.exp, (arr,), {}, {})

# Case 3: interior-tracked (recipe set, rg=True)
t3 = MiniTensor(arr.copy(), requires_grad=True)
t3.recipe = fake_recipe

# Case 4: interior-frozen (recipe set, rg=False)
t4 = MiniTensor(arr.copy(), requires_grad=False)
t4.recipe = fake_recipe

results = [is_leaf(t1), is_leaf(t2), is_leaf(t3), is_leaf(t4)]
labels  = ['leaf-trainable', 'leaf-frozen', 'interior-tracked', 'interior-frozen']
for label, result in zip(labels, results):
    print(f'{label}: is_leaf = {result}')

assert results == [True, False, False, False]
print('All correct!')